In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/size_only/validation/ibm'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Offline ABC–IPM Point Estimates

This notebook compares exact-GP and DeepRV simulation using the **same 5,000 parameter combinations**, the **same observed summaries**, and **common exact-GP thresholds**. It uses the 50 saved simulations per parameter combination; it does not run new IBMs or full ABC-PMC.

For parameter combination $i$ and replicate $r$, the 16 selected year-summary discrepancies are combined as

$$D_{ir}=\sqrt{\sum_j\left(s_{irj}/\operatorname{MAD}_j\right)^2}.$$

The MADs and thresholds are calculated from exact-GP simulations only. At a common threshold $\epsilon$, the weight of parameter combination $i$ is proportional to its estimated acceptance probability, $\hat p_i=R^{-1}\sum_r I(D_{ir}\leq\epsilon)$. One IPM is built for each parameter combination, then the exact-GP and DeepRV weights are applied to those same IPM metrics. Results are point estimates only: there is no bootstrap.


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import sys

import gpflow
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from gpflow.conditionals.util import base_conditional_with_lm
from scipy.stats import norm

def find_project_directory(start):
    for directory in (start, *start.parents):
        if (directory / "pyproject.toml").is_file():
            return directory
    raise FileNotFoundError("Could not find the dl4bi project directory")

project_directory = PROJECT_ROOT
sys.path.insert(0, str(project_directory))
from experiments.soay.common.ipm_utils import calculate_ipm_metrics

result_directory = GENERATED / 'soay/size_only/validation/ibm'
design_file = result_directory / "benchmark_design.npz"
samples_file = GENERATED / 'soay/size_only/inference/gpdf_gp_samples.npz'
data_file = GENERATED / 'soay/size_only/data/gp_ibm_dataset.csv'
true_metrics_file = GENERATED / 'soay/size_only/data/true_gp_metrics.npz'
distance_cache_file = result_directory / "offline_scalar_distances.npz"
prediction_cache_file = result_directory / "offline_vital_rate_predictions.npz"
ipm_cache_file = result_directory / "offline_ipm_metrics.npz"
point_estimate_file = result_directory / "offline_ipm_point_estimates.csv"
weight_file = result_directory / "offline_rejection_weights.npz"

selected_summaries = ["kl_next_size_distribution", "chi2_total_survivors_by_size", "chi2_total_breeders_by_size", "chi2_total_individuals_next_size"]
threshold_quantiles = np.array([0.50, 0.20, 0.10, 0.05, 0.01])
maximum_load_workers = 8
lower_size, upper_size, number_mesh_points = 1.7, 3.4, 250
rate_columns = {"growth": 0, "surv": 1, "repr": 2, "rcsz": 3}
method_colors = {"Exact GP": "#0072B2", "DeepRV": "#D55E00"}
plt.style.use("seaborn-v0_8-white")
gpflow.config.set_default_float(np.float64)
print(f"Project Directory: {project_directory}")


## Load And Validate The Fixed Design

Each row of `parameter_indices` is one modular parameter combination: one joint posterior index for growth, survival, reproduction, and recruit size. The same row was used for its 50 exact-GP simulations and 50 DeepRV simulations.


In [ ]:
with np.load(design_file) as design:
    parameter_indices = design["parameter_indices"].copy()
    summary_names = design["summary_names"].astype(str)
    years = design["years"].copy()
    number_parameters = int(design["number_parameters"].item())
    number_replicates = int(design["number_replicates"].item())
    recruitment_probability = float(design["recruit_probability"].item())
    sample_hash = str(design["samples_sha256"].item())

if parameter_indices.shape != (number_parameters, 4):
    raise ValueError("Unexpected parameter-index shape")
if not set(selected_summaries).issubset(summary_names):
    raise ValueError("A selected summary is absent from the benchmark design")
summary_indices = np.array([np.flatnonzero(summary_names == name).item() for name in selected_summaries])
print(f"Parameter Combinations: {number_parameters:,}")
print(f"Replicates Per Combination And Method: {number_replicates}")
print(f"Years: {years.tolist()}")
print(f"Selected Summaries: {selected_summaries}")


## Construct Scalar Distances

The first run opens every raw chunk and writes a compact cache. This can take several minutes because there are 2,500 small files per method. Later runs load the compact cache directly. Infinite discrepancies remain infinite, so a trajectory with an unusable year cannot pass a finite threshold.


In [ ]:
def read_selected_chunk(arguments):
    chunk_path, indices = arguments
    with np.load(chunk_path) as chunk:
        return chunk["parameter_id"].astype(int), chunk["replicate_id"].astype(int), chunk["distances"][:, :, indices]

def load_selected_summaries(method_directory):
    shape = (number_parameters, number_replicates, len(years), len(summary_indices))
    values = np.full(shape, np.nan)
    seen = np.zeros(shape[:2], dtype=bool)
    chunk_paths = sorted(method_directory.glob("chunk_*.npz"))
    if not chunk_paths:
        raise FileNotFoundError(f"No chunks found in {method_directory}")
    arguments = ((path, summary_indices) for path in chunk_paths)
    with ThreadPoolExecutor(max_workers=maximum_load_workers) as pool:
        for chunk_number, (parameter_id, replicate_id, distances) in enumerate(pool.map(read_selected_chunk, arguments), start=1):
            if np.any(seen[parameter_id, replicate_id]):
                raise ValueError(f"Duplicate result in {chunk_paths[chunk_number - 1]}")
            values[parameter_id, replicate_id] = distances
            seen[parameter_id, replicate_id] = True
            if chunk_number % 250 == 0 or chunk_number == len(chunk_paths):
                print(f"{method_directory.name}: {chunk_number:,}/{len(chunk_paths):,} Chunks", flush=True)
    if not seen.all():
        raise ValueError(f"{method_directory} is missing {(~seen).sum():,} results")
    return values

def scalar_distances(values, mad):
    return np.sqrt(np.sum((values / mad[None, None, :, :]) ** 2, axis=(2, 3)))

if distance_cache_file.exists():
    with np.load(distance_cache_file) as cache:
        cached_summaries = cache["selected_summaries"].astype(str).tolist()
        if cached_summaries != selected_summaries:
            raise ValueError("The distance cache uses different summaries; delete it and rerun this cell")
        exact_distances = cache["exact_distances"].copy()
        deeprv_distances = cache["deeprv_distances"].copy()
        component_mad = cache["component_mad"].copy()
    print(f"Loaded {distance_cache_file.name}")
else:
    exact_summaries = load_selected_summaries(result_directory / "exact_gp")
    finite_exact = np.where(np.isfinite(exact_summaries), exact_summaries, np.nan)
    component_median = np.nanmedian(finite_exact, axis=(0, 1))
    component_mad = np.nanmedian(np.abs(finite_exact - component_median[None, None, :, :]), axis=(0, 1))
    if np.any(~np.isfinite(component_mad)) or np.any(component_mad <= 0):
        raise ValueError("At least one selected year-summary component has an invalid MAD")
    exact_distances = scalar_distances(exact_summaries, component_mad)
    del exact_summaries, finite_exact
    deeprv_summaries = load_selected_summaries(result_directory / "deeprv")
    deeprv_distances = scalar_distances(deeprv_summaries, component_mad)
    del deeprv_summaries
    np.savez_compressed(distance_cache_file, exact_distances=exact_distances, deeprv_distances=deeprv_distances, component_mad=component_mad, selected_summaries=np.array(selected_summaries))
    print(f"Saved {distance_cache_file.name}")

expected_shape = (number_parameters, number_replicates)
if exact_distances.shape != expected_shape or deeprv_distances.shape != expected_shape:
    raise ValueError("Unexpected scalar-distance shape")
mad_table = pd.DataFrame(component_mad, index=[f"Year {year}" for year in years], columns=selected_summaries)
display(mad_table.round(4))


## Common Thresholds And Rejection Weights

Each threshold is a quantile of all $5{,}000\times50$ exact-GP scalar distances, including infinite distances from unusable trajectories. The same numerical threshold is then applied to both methods. Because the 5,000 design points have equal prior mass, their normalized acceptance frequencies are the rejection-ABC weights.


In [ ]:
if np.isnan(exact_distances).any() or np.isnan(deeprv_distances).any():
    raise ValueError("NaN scalar distances were found")
if np.mean(np.isfinite(exact_distances)) <= threshold_quantiles.max():
    raise ValueError("Too few finite exact-GP distances for the requested thresholds")

thresholds = np.quantile(exact_distances.ravel(), threshold_quantiles)
weights = {}
diagnostic_rows = []
for quantile, threshold in zip(threshold_quantiles, thresholds):
    for method, distances in {"Exact GP": exact_distances, "DeepRV": deeprv_distances}.items():
        acceptance_probability = np.mean(distances <= threshold, axis=1)
        if acceptance_probability.sum() == 0:
            raise ValueError(f"No {method} simulations passed the {quantile:.0%} threshold")
        method_weights = acceptance_probability / acceptance_probability.sum()
        weights[(quantile, method)] = method_weights
        diagnostic_rows.append({"Threshold Quantile": quantile, "Threshold": threshold, "Method": method, "Overall Acceptance Rate": acceptance_probability.mean(), "Active Parameters": np.count_nonzero(acceptance_probability), "ESS": 1 / np.sum(method_weights ** 2)})

weight_diagnostics = pd.DataFrame(diagnostic_rows)
display(weight_diagnostics.assign(**{"Threshold Quantile": lambda frame: frame["Threshold Quantile"].map(lambda value: f"{value:.0%}"), "Overall Acceptance Rate": lambda frame: frame["Overall Acceptance Rate"].map(lambda value: f"{value:.2%}")}))


## Build One IPM Per Parameter Combination

This section matches the GP IPM construction in `experiments/soay/size_only/abc/Make_IPM_deeprv.ipynb`. The stored MCMC function values and hyperparameters define each fitted vital-rate curve on the 250-point mesh. The fixed recruitment probability is loaded from the benchmark design, so it is identical to the probability used in both sets of saved IBMs.

The DeepRV decoder is not called here: its effect is already represented by the DeepRV acceptance weights. Vital-rate predictions and IPM outputs are cached because this section is computationally expensive.


In [ ]:
data = pd.read_csv(data_file)
with np.load(samples_file) as saved:
    gp_samples = {name: saved[name].copy() for name in saved.files}
with np.load(true_metrics_file) as saved:
    true_metrics = {name: saved[name].copy() for name in saved.files}

training_inputs = {"surv": data[["z"]].to_numpy(np.float64), "repr": data[["z", "Repr"]].dropna()[["z"]].to_numpy(np.float64), "growth": data[["z", "z1"]].dropna()[["z"]].to_numpy(np.float64), "rcsz": data[["z", "Rcsz"]].dropna()[["z"]].to_numpy(np.float64)}
mesh_width = (upper_size - lower_size) / number_mesh_points
mesh = lower_size + (np.arange(number_mesh_points) + 0.5) * mesh_width
print({rate: len(values) for rate, values in training_inputs.items()})


## Particle-Level Acceptance Agreement

Each point below is one complete, matched parameter combination. For each exact-GP distance quantile, its coordinates are the fractions of that particle's 50 simulations that pass the same numerical distance threshold under exact GP and DeepRV. The dashed line denotes equal acceptance probability. Because there are 50 replicates, both coordinates occur in increments of 0.02.


In [ ]:
acceptance_quantiles = np.array([0.20, 0.10, 0.05, 0.01])
figure, axes = plt.subplots(2, 2, figsize=(5.906, 5.4), sharex=True, sharey=True, constrained_layout=True)

for axis, quantile in zip(axes.ravel(), acceptance_quantiles):
    threshold_index = np.flatnonzero(np.isclose(threshold_quantiles, quantile)).item()
    threshold = thresholds[threshold_index]
    exact_acceptance = np.mean(exact_distances <= threshold, axis=1)
    deeprv_acceptance = np.mean(deeprv_distances <= threshold, axis=1)
    difference = deeprv_acceptance - exact_acceptance
    correlation = np.corrcoef(exact_acceptance, deeprv_acceptance)[0, 1]

    axis.scatter(exact_acceptance, deeprv_acceptance, s=7, color=method_colors["Exact GP"], alpha=0.15, edgecolors="none", rasterized=True)
    axis.plot([0, 1], [0, 1], color="#4D4D4D", linestyle="--", linewidth=1.0)
    axis.set_title(rf"${quantile:.2f}$ Quantile", fontsize=11)
    axis.set_xlim(-0.02, 1.02)
    axis.set_ylim(-0.02, 1.02)
    axis.set_xticks([0, 0.5, 1])
    axis.set_yticks([0, 0.5, 1])
    axis.set_aspect("equal", adjustable="box")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(labelsize=9, direction="out")

figure.supxlabel("Exact-GP Acceptance Probability", fontsize=11)
figure.supylabel("DeepRV Acceptance Probability", fontsize=11)
figure.savefig(result_directory / "acceptance_probability_agreement.pdf", bbox_inches="tight")
figure.savefig(result_directory / "acceptance_probability_agreement.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
class MCMCConditionedGP:
    def __init__(self, x_train, mu_train, variance, lengthscale, intercept, likelihood, jitter=5e-4):
        dtype = gpflow.default_float()
        self.x_train = tf.convert_to_tensor(x_train, dtype=dtype)
        self.intercept = tf.convert_to_tensor(intercept, dtype=dtype)
        self.likelihood = likelihood
        centered_values = np.asarray(mu_train, dtype=np.float64).reshape(-1, 1) - float(intercept)
        self.centered_values = tf.convert_to_tensor(centered_values, dtype=dtype)
        self.kernel = gpflow.kernels.SquaredExponential(variance=float(variance), lengthscales=float(lengthscale))
        covariance = self.kernel(self.x_train, self.x_train) + tf.cast(jitter, dtype) * tf.eye(len(x_train), dtype=dtype)
        self.cholesky = tf.linalg.cholesky(covariance)

    def predict_f(self, x):
        x = tf.reshape(tf.convert_to_tensor(x, dtype=gpflow.default_float()), (-1, 1))
        cross_covariance = self.kernel(self.x_train, x)
        mean, variance = base_conditional_with_lm(cross_covariance, self.cholesky, self.kernel.K_diag(x), self.centered_values, full_cov=False, white=False)
        return mean + self.intercept, variance

    def predict_y(self, x):
        mean, variance = self.predict_f(x)
        x = tf.reshape(tf.convert_to_tensor(x, dtype=gpflow.default_float()), (-1, 1))
        return self.likelihood.predict_mean_and_var(x, mean, variance)

def predict_vital_rate(rate, sample_index):
    if rate in {"surv", "repr"}:
        likelihood = gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid)
    else:
        likelihood = gpflow.likelihoods.Gaussian(variance=float(gp_samples[f"{rate}_noise"][sample_index]) ** 2)
    model = MCMCConditionedGP(training_inputs[rate], gp_samples[f"{rate}_mu"][sample_index], gp_samples[f"{rate}_var"][sample_index], gp_samples[f"{rate}_ls"][sample_index], gp_samples[f"{rate}_int"][sample_index], likelihood)
    mean, variance = model.predict_y(mesh)
    return mean.numpy().ravel(), variance.numpy().ravel()


In [ ]:
if prediction_cache_file.exists():
    with np.load(prediction_cache_file) as cache:
        if str(cache["sample_hash"].item()) != sample_hash or int(cache["number_mesh_points"].item()) != number_mesh_points:
            raise ValueError("The vital-rate cache does not match this design or mesh")
        vital_rate_predictions = {rate: {"mean": cache[f"{rate}_mean"].copy(), "variance": cache[f"{rate}_variance"].copy()} for rate in rate_columns}
    print(f"Loaded {prediction_cache_file.name}")
else:
    archive_size = len(gp_samples["surv_int"])
    vital_rate_predictions = {rate: {"mean": np.full((archive_size, number_mesh_points), np.nan), "variance": np.full((archive_size, number_mesh_points), np.nan)} for rate in rate_columns}
    for rate, column in rate_columns.items():
        used_indices = np.unique(parameter_indices[:, column])
        for position, sample_index in enumerate(used_indices, start=1):
            mean, variance = predict_vital_rate(rate, sample_index)
            vital_rate_predictions[rate]["mean"][sample_index] = mean
            vital_rate_predictions[rate]["variance"][sample_index] = variance
            if position % 250 == 0 or position == len(used_indices):
                print(f"{rate}: {position:,}/{len(used_indices):,} Curves", flush=True)
    cache_values = {f"{rate}_{name}": values for rate, rate_values in vital_rate_predictions.items() for name, values in rate_values.items()}
    np.savez_compressed(prediction_cache_file, **cache_values, sample_hash=np.array(sample_hash), number_mesh_points=np.array(number_mesh_points))
    print(f"Saved {prediction_cache_file.name}")


In [ ]:
def make_kernel(predictions):
    survival = np.clip(predictions["surv"], 0, 1)
    reproduction = np.clip(predictions["repr"], 0, 1)
    growth = norm.pdf(mesh[:, None], loc=predictions["growth"][None, :], scale=np.sqrt(predictions["growth_var"])[None, :])
    recruit_size = norm.pdf(mesh[:, None], loc=predictions["rcsz"][None, :], scale=np.sqrt(predictions["rcsz_var"])[None, :])
    survival_kernel = mesh_width * survival[None, :] * growth
    fecundity = survival * reproduction * 0.5 * recruitment_probability
    fecundity_kernel = mesh_width * fecundity[None, :] * recruit_size
    survival_kernel *= np.divide(survival, survival_kernel.sum(axis=0), out=np.zeros_like(survival), where=survival_kernel.sum(axis=0) > 0)
    fecundity_kernel *= np.divide(fecundity, fecundity_kernel.sum(axis=0), out=np.zeros_like(fecundity), where=fecundity_kernel.sum(axis=0) > 0)
    return survival_kernel + fecundity_kernel

if ipm_cache_file.exists():
    with np.load(ipm_cache_file) as cache:
        if str(cache["sample_hash"].item()) != sample_hash or int(cache["number_mesh_points"].item()) != number_mesh_points:
            raise ValueError("The IPM cache does not match this design or mesh")
        ipm_metrics = {name: cache[name].copy() for name in ["lambda", "omega", "omega_rep", "gamma"]}
    print(f"Loaded {ipm_cache_file.name}")
else:
    ipm_metrics = {"lambda": np.empty(number_parameters), "omega": np.empty((number_parameters, number_mesh_points)), "omega_rep": np.empty((number_parameters, number_mesh_points)), "gamma": np.empty((number_parameters, number_mesh_points))}
    for parameter_id, indices in enumerate(parameter_indices):
        predictions = {"growth": vital_rate_predictions["growth"]["mean"][indices[0]], "growth_var": vital_rate_predictions["growth"]["variance"][indices[0]], "surv": vital_rate_predictions["surv"]["mean"][indices[1]], "repr": vital_rate_predictions["repr"]["mean"][indices[2]], "rcsz": vital_rate_predictions["rcsz"]["mean"][indices[3]], "rcsz_var": vital_rate_predictions["rcsz"]["variance"][indices[3]]}
        metrics = calculate_ipm_metrics(make_kernel(predictions), predictions["repr"])
        for name in ipm_metrics:
            ipm_metrics[name][parameter_id] = metrics[name]
        if (parameter_id + 1) % 250 == 0 or parameter_id + 1 == number_parameters:
            print(f"IPMs: {parameter_id + 1:,}/{number_parameters:,}", flush=True)
    np.savez_compressed(ipm_cache_file, **ipm_metrics, sample_hash=np.array(sample_hash), number_mesh_points=np.array(number_mesh_points))
    print(f"Saved {ipm_cache_file.name}")


## Weighted Point Estimates

The losses are calculated for each IPM first and then averaged with the rejection weights. For example, lambda MSE is $\sum_i w_i(\lambda_i-\lambda_{\mathrm{true}})^2$; it is not the squared error of the weighted mean lambda. Lower is better for the three loss metrics, while higher is better for reproductive-value cosine similarity.


In [ ]:
def normalise_rows(values, epsilon=1e-12):
    values = np.clip(np.asarray(values, dtype=float), epsilon, None)
    return values / values.sum(axis=-1, keepdims=True)

true_lambda = float(true_metrics["lambda"])
true_omega = normalise_rows(true_metrics["omega"])
true_omega_rep = normalise_rows(true_metrics["omega_rep"])
true_gamma = np.asarray(true_metrics["gamma"], dtype=float)
omega = normalise_rows(ipm_metrics["omega"])
omega_rep = normalise_rows(ipm_metrics["omega_rep"])
gamma = np.asarray(ipm_metrics["gamma"], dtype=float)
particle_scores = {"Lambda MSE": (ipm_metrics["lambda"] - true_lambda) ** 2, "Stable State KL": np.sum(true_omega * np.log(true_omega / omega), axis=1), "Reproductive Value Cosine Similarity": np.sum(gamma * true_gamma, axis=1) / (np.linalg.norm(gamma, axis=1) * np.linalg.norm(true_gamma)), "Reproductive Stable Distribution KL": np.sum(true_omega_rep * np.log(true_omega_rep / omega_rep), axis=1)}

result_rows = []
for quantile, threshold in zip(threshold_quantiles, thresholds):
    for method in method_colors:
        method_weights = weights[(quantile, method)]
        lambda_mean = np.sum(method_weights * ipm_metrics["lambda"])
        row = {"Threshold Quantile": quantile, "Threshold": threshold, "Method": method, "Lambda Mean": lambda_mean, "Lambda Standard Deviation": np.sqrt(np.sum(method_weights * (ipm_metrics["lambda"] - lambda_mean) ** 2))}
        row.update({name: np.sum(method_weights * values) for name, values in particle_scores.items()})
        result_rows.append(row)

point_estimates = pd.DataFrame(result_rows).merge(weight_diagnostics, on=["Threshold Quantile", "Threshold", "Method"], validate="one_to_one")
display(point_estimates.round(6))


In [ ]:
comparison_metrics = ["Lambda MSE", "Stable State KL", "Reproductive Value Cosine Similarity", "Reproductive Stable Distribution KL"]
method_difference = point_estimates.pivot(index="Threshold Quantile", columns="Method", values=comparison_metrics)
method_difference = method_difference.xs("DeepRV", axis=1, level=1) - method_difference.xs("Exact GP", axis=1, level=1)
method_difference.index = method_difference.index.map(lambda value: f"{value:.0%}")
method_difference.index.name = "Threshold Quantile"
print("DeepRV Minus Exact GP")
display(method_difference.round(6))


In the difference table, a negative value favours DeepRV for the three loss metrics. A positive value favours DeepRV for reproductive-value cosine similarity. These are descriptive point differences, not uncertainty intervals or significance tests.


## Weighted Lambda Distributions

These panels compare the exact-GP and DeepRV weighted lambda distributions at every common threshold. Both methods use the same 5,000 lambda values; only their rejection-ABC weights differ. The same horizontal range and kernel-density bandwidth are used throughout, making the panels directly comparable. Close distributions indicate that replacing exact GP simulation with DeepRV has little effect on the inferred lambda distribution.


In [ ]:
lambda_values = ipm_metrics["lambda"]
active = np.zeros(number_parameters, dtype=bool)
for method_weights in weights.values():
    active |= method_weights > 0
plot_values = lambda_values[active]
plot_padding = 0.03 * np.ptp(plot_values)
lambda_grid = np.linspace(plot_values.min() - plot_padding, plot_values.max() + plot_padding, 400)
bandwidth = 1.06 * np.std(plot_values, ddof=1) * len(plot_values) ** (-1 / 5)

def weighted_density(values, method_weights):
    scaled_difference = (lambda_grid[:, None] - values[None, :]) / bandwidth
    return norm.pdf(scaled_difference) @ method_weights / bandwidth

figure, axes = plt.subplots(2, 3, figsize=(10.8, 6.0), sharex=True, sharey=True)
for axis, quantile in zip(axes.ravel(), threshold_quantiles):
    for row, (method, color) in enumerate(method_colors.items()):
        method_weights = weights[(quantile, method)]
        density = weighted_density(lambda_values, method_weights)
        lambda_mean = np.sum(method_weights * lambda_values)
        axis.fill_between(lambda_grid, density, color=color, alpha=0.16)
        axis.plot(lambda_grid, density, color=color, linewidth=1.4, alpha=0.9, label=method)
        axis.axvline(lambda_mean, color=color, linestyle="--", linewidth=1.2, alpha=0.85)
        axis.text(0.03, 0.95 - 0.10 * row, f"{method} Mean = {lambda_mean:.5f}", transform=axis.transAxes, color=color, fontsize=9, va="top")
    axis.set_title(f"{quantile:.0%} Threshold", fontsize=11)
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(labelsize=9, direction="out")
axes.ravel()[-1].set_visible(False)
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.subplots_adjust(left=0.08, right=0.98, bottom=0.10, top=0.82, wspace=0.04, hspace=0.18)
figure.suptitle("Weighted Lambda Distributions Across Thresholds", fontsize=16, y=0.98)
figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.925), ncol=2, frameon=False)
figure.supxlabel(r"Population Growth Rate $\lambda$", fontsize=14, y=0.02)
figure.supylabel("Weighted Density", fontsize=14, x=0.02)
#figure.savefig(result_directory / "offline_weighted_lambda_distributions_all_thresholds.pdf", bbox_inches="tight")
#figure.savefig(result_directory / "offline_weighted_lambda_distributions_all_thresholds.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
def weighted_density(grid, values, method_weights, bandwidth):
    differences = (grid[:, None] - values[None, :]) / bandwidth
    return norm.pdf(differences) @ method_weights / bandwidth

def density_grid(values, points=400):
    padding = max(0.03 * np.ptp(values), 1e-8)
    grid = np.linspace(values.min() - padding, values.max() + padding, points)
    bandwidth = max(1.06 * np.std(values, ddof=1) * len(values) ** (-1 / 5), 1e-8)
    return grid, bandwidth

method_styles = {
    "Exact GP": {"color": "#0072B2", "linestyle": "-"},
    "DeepRV": {"color": "#D55E00", "linestyle": "-"},
}

lambda_values = ipm_metrics["lambda"]

active = np.zeros(number_parameters, dtype=bool)
for method_weights in weights.values():
    active |= method_weights > 0

lambda_grid, lambda_bandwidth = density_grid(lambda_values[active])

plt.style.use("seaborn-v0_8-white")
figure, axes = plt.subplots(2, 3, figsize=(5.906, 5.2), sharex=True, sharey=True, constrained_layout=True)

for axis, quantile in zip(axes.ravel(), threshold_quantiles):
    for method, style in method_styles.items():
        method_weights = weights[(quantile, method)]
        density = weighted_density(lambda_grid, lambda_values, method_weights, lambda_bandwidth)
        lambda_mean = np.sum(method_weights * lambda_values)

        axis.fill_between(lambda_grid, density, color=style["color"], alpha=0.10)
        axis.plot(lambda_grid, density, color=style["color"], linestyle=style["linestyle"], linewidth=1.5, label=method)
        axis.axvline(lambda_mean, color=style["color"], linestyle=":", linewidth=1.2)

    axis.set_title(f"{quantile:.0%} Threshold", fontsize=11)
    axis.tick_params(labelsize=9, direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

axes.ravel()[-1].remove()

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="outside upper center", ncol=2, frameon=False, fontsize=9)
figure.supxlabel(r"Population Growth Rate $\lambda$", fontsize=11)
figure.supylabel("Weighted Density", fontsize=11)

#figure.savefig(result_directory / "offline_weighted_lambda_distributions_all_thresholds.pdf", bbox_inches="tight")
#figure.savefig(result_directory / "offline_weighted_lambda_distributions_all_thresholds.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
metrics_to_plot = ["Stable State KL", "Reproductive Value Cosine Similarity", "Reproductive Stable Distribution KL"]

def weighted_kde(grid, values, method_weights, bandwidth):
    scaled_difference = (grid[:, None] - values[None, :]) / bandwidth
    return norm.pdf(scaled_difference) @ method_weights / bandwidth

for metric in metrics_to_plot:
    metric_values = particle_scores[metric]
    active = np.zeros(len(metric_values), dtype=bool)

    for method_weights in weights.values():
        active |= method_weights > 0

    plot_values = metric_values[active]
    plot_padding = 0.03 * np.ptp(plot_values)
    lower_limit = plot_values.min() - plot_padding
    upper_limit = plot_values.max() + plot_padding

    if "KL" in metric:
        lower_limit = max(0, lower_limit)
    else:
        lower_limit = max(-1, lower_limit)
        upper_limit = min(1, upper_limit)

    metric_grid = np.linspace(lower_limit, upper_limit, 400)
    bandwidth = 1.06 * np.std(plot_values, ddof=1) * len(plot_values) ** (-1 / 5)

    figure, axes = plt.subplots(2, 3, figsize=(10.8, 6.0), sharex=True, sharey=True)

    for axis, quantile in zip(axes.ravel(), threshold_quantiles):
        for row, (method, color) in enumerate(method_colors.items()):
            method_weights = weights[(quantile, method)]
            density = weighted_kde(metric_grid, metric_values, method_weights, bandwidth)
            weighted_mean = np.sum(method_weights * metric_values)

            axis.fill_between(metric_grid, density, color=color, alpha=0.16)
            axis.plot(metric_grid, density, color=color, linewidth=1.4, alpha=0.9, label=method)
            axis.axvline(weighted_mean, color=color, linestyle="--", linewidth=1.2, alpha=0.85)
            axis.text(0.03, 0.95 - 0.10 * row, f"{method} Mean = {weighted_mean:.4g}", transform=axis.transAxes, color=color, fontsize=9, va="top")

        axis.set_title(f"{quantile:.0%} Threshold", fontsize=11)
        axis.grid(False)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
        axis.tick_params(labelsize=9, direction="out")

    axes.ravel()[-1].set_visible(False)
    handles, labels = axes[0, 0].get_legend_handles_labels()

    figure.subplots_adjust(left=0.08, right=0.98, bottom=0.10, top=0.82, wspace=0.04, hspace=0.18)
    figure.suptitle(f"Weighted {metric} Distributions Across Common Thresholds", fontsize=16, y=0.98)
    figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.925), ncol=2, frameon=False)
    figure.supxlabel(metric, fontsize=14, y=0.02)
    figure.supylabel("Weighted Density", fontsize=14, x=0.02)

    file_name = metric.lower().replace(" ", "_")
    #figure.savefig(result_directory / f"offline_weighted_{file_name}_distributions.pdf", bbox_inches="tight")
    #figure.savefig(result_directory / f"offline_weighted_{file_name}_distributions.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
def weighted_density(grid, values, method_weights, bandwidth):
    differences = (grid[:, None] - values[None, :]) / bandwidth
    return norm.pdf(differences) @ method_weights / bandwidth

def density_grid(values, points=400):
    padding = max(0.03 * np.ptp(values), 1e-8)
    grid = np.linspace(values.min() - padding, values.max() + padding, points)
    bandwidth = max(1.06 * np.std(values, ddof=1) * len(values) ** (-1 / 5), 1e-8)
    return grid, bandwidth

method_styles = {
    "Exact GP": {"color": "#0072B2", "linestyle": "-"},
    "DeepRV": {"color": "#D55E00", "linestyle": "-"},
}

metrics_to_plot = {
    "Lambda MSE": particle_scores["Lambda MSE"],
    "Stable State KL": particle_scores["Stable State KL"],
    "Reproductive Value\nCosine Similarity": particle_scores["Reproductive Value Cosine Similarity"],
    "Reproductive Stable\nDistribution KL": particle_scores["Reproductive Stable Distribution KL"],
}

selected_quantile = 0.01
active = (weights[(selected_quantile, "Exact GP")] > 0) | (weights[(selected_quantile, "DeepRV")] > 0)

plt.style.use("seaborn-v0_8-white")
figure, axes = plt.subplots(2, 2, figsize=(5.906, 4.8), constrained_layout=True)

for axis, (metric, metric_values) in zip(axes.ravel(), metrics_to_plot.items()):
    metric_grid, bandwidth = density_grid(metric_values[active])

    if "KL" in metric or "MSE" in metric:
        metric_grid = np.linspace(max(0, metric_grid.min()), metric_grid.max(), len(metric_grid))
    else:
        metric_grid = np.linspace(max(-1, metric_grid.min()), min(1, metric_grid.max()), len(metric_grid))

    for method, style in method_styles.items():
        method_weights = weights[(selected_quantile, method)]
        density = weighted_density(metric_grid, metric_values, method_weights, bandwidth)
        weighted_mean = np.sum(method_weights * metric_values)

        axis.fill_between(metric_grid, density, color=style["color"], alpha=0.10)
        axis.plot(metric_grid, density, color=style["color"], linestyle=style["linestyle"], linewidth=1.5, label=method)
        axis.axvline(weighted_mean, color=style["color"], linestyle=":", linewidth=1.2)

    axis.set_xlabel(metric, fontsize=9)
    axis.tick_params(labelsize=9, direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="outside upper center", ncol=2, frameon=False, fontsize=9)
figure.supylabel("Weighted Density", fontsize=11)

#figure.savefig(result_directory / "offline_weighted_ipm_metrics_1_percent.pdf", bbox_inches="tight")
#figure.savefig(result_directory / "offline_weighted_ipm_metrics_1_percent.png", dpi=300, bbox_inches="tight")

plt.show()

## Save Results

The table and normalized weights are saved so later plots do not need to reopen the raw chunks or rebuild IPMs.


In [ ]:
point_estimates.to_csv(point_estimate_file, index=False)
weight_values = {f"{method.lower().replace(' ', '_')}_{int(100 * quantile):02d}": values for (quantile, method), values in weights.items()}
np.savez_compressed(weight_file, **weight_values, parameter_indices=parameter_indices, threshold_quantiles=threshold_quantiles, thresholds=thresholds)
print(f"Saved {point_estimate_file.name}")
print(f"Saved {weight_file.name}")
